# krig - Intro Notebook

Declarative device SDK: describe properties, compose pipelines, time-travel through state.

For a local checkout, run `./gradlew publishToMavenLocal` first, then load the sibling descriptor:
`%use @file[krig.json]`.
After krig is published as a Kotlin Notebook library, this cell can become `%use krig`.

In [ ]:
%use @file[krig.json]
// Covers: Device, Timestamped, ObservedValue, DeviceOutcome, LifecycleState,
//         Timeline, DeviceMessage, Meta, Magix, simulation, DSL, coroutines.

## 1. Inline device - read / write / actions

In [ ]:
val thermo = runBlocking {
    device("thermo", scriptContext()) {
        mutableProperty("temperature", initial = 22.0)
        mutableProperty("setpoint", initial = 20.0)
        action("reset") { okUnit() }
    }
}
thermo

In [ ]:
// DeviceOutcome - faults as values, no try/catch
val outcome = runBlocking { thermo.readPropertyOutcome("temperature".asName()) }
outcome

In [ ]:
// Write through the Meta/control-plane boundary, re-read
runBlocking {
    thermo.writeProperty("temperature".asName(), metaOf(25.0))
    thermo.readProperty("temperature".asName())
}

## 2. Property history

In [ ]:
import kotlinx.coroutines.flow.toList

runBlocking {
    thermo.writeProperty("temperature".asName(), metaOf(30.0))
    thermo.writeProperty("temperature".asName(), metaOf(31.0))

    val history = thermo.propertyHistory("temperature".asName(), MetaConverter.double)
    history.flowHistory().toList()
}

## 3. Dynamic hub - attach, detach, reconcile

In [ ]:
val hubCtx = Context("hub-demo")
val hub = MutableCompositeDevice("hub".asName(), hubCtx)

runBlocking {
    val child = device("child", hubCtx) {
        mutableProperty("ready", initial = true)
    }
    hub.attach("child".asName(), child)
}

println("Children: ${hub.children.keys}")
hub

## 4. Timeline - mergeable event streams

In [ ]:
// Timeline wraps a Flow<DeviceMessage> with merge combinators for multi-device views
val tl = thermo.timeline()
tl

In [ ]:
// Event log: cold, replayable source for time-travel and counterfactuals
val log = thermo.eventLog()
log

## 5. Simulation - virtual time

In [ ]:
// Simulation symbols are loaded by krig.json through krig-jupyter.
// Available here: DeterministicScheduler, ProcessDsl, Resource, Signal, SimulationSession.

In [ ]:
import kotlinx.coroutines.launch
import kotlin.time.Duration.Companion.seconds

val sched = DeterministicScheduler()  // initialTimeMs defaults to 0
val scope = CoroutineScope(sched.asDispatcher())

// Process DSL: hold, waitUntil, request - coroutine-native, no yield sentinels
scope.process("ramp") {
    hold(1.seconds)
    hold(2.seconds)
}

// Advance virtual time 5 seconds - coroutines scheduled in that window all run
runBlocking { sched.advanceBy(5.seconds) }
sched.currentTimeMs

## 6. In-memory storage

In [ ]:
val storage = InMemoryDeviceMessageStorage()

// Storage-backed PropertyHistory - replays recorded PropertyChangedMessages
val storageHistory = storage.propertyHistory(
    "thermo".asName(),
    "temperature".asName(),
    MetaConverter.double,
)
storageHistory

In [ ]:
// The history is lazy - it reads from storage on each flow subscription
runBlocking {
    storage.write(
        PropertyChangedMessage(
            time = Clock.System.now(),
            property = "temperature".asName(),
            value = metaOf(42.0),
            sourceDevice = "thermo".asName(),
            targetDevice = null,
        )
    )
    storageHistory.flowHistory().toList()
}